# 02 — Results

Read-only view over `outputs/`. Run `extract.py`, `assistant_axis.py`,
`probes.py` and `plot.py` first.

Nothing is computed here that a script does not also compute, so no result
depends on cell execution order.

In [ ]:
import sys, pathlib, json

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
from IPython.display import Image, display
import config, personas

missing = [p.name for p in (config.ACTIVATIONS_INFO, config.AXIS_RANKING_JSON,
                            config.PROBE_RESULTS_JSON) if not p.exists()]
if missing:
    raise SystemExit(f"missing {missing} — run the pipeline first")

info = json.loads(config.ACTIVATIONS_INFO.read_text())
axis = json.loads(config.AXIS_RANKING_JSON.read_text())
probe = json.loads(config.PROBE_RESULTS_JSON.read_text())
BEST = str(probe["best_layer"])
CV = probe["cv"]

print(f"model      {info['model']} ({info['preset']}, {info['dtype']})")
print(f"acts       {tuple(info['shape'])}  layers {info['layers']}")
print(f"pooling    {info['pooling']}")
print(f"cv         {CV['n_folds']}-fold over topics, seed {CV['seed']} — "
      f"{[len(f) for f in CV['folds']]} topics held out per fold")
print(f"best layer {BEST}  (selected on {probe['best_layer_selected_on']})")
print(f"chance     {probe['chance']:.3f}   classes {probe['n_classes']}")
print(f"axis       {axis['anchor_persona']} (0) -> {axis['far_persona']} (1), "
      f"control {axis['control_persona']}")

## The headline

Probe accuracy against measured distance along the assistant axis. `bare_template`
is a **control**, not a persona — it is drawn in its own colour and marker, and it
never contributed to defining the axis.

Each point is the mean over the topic folds and the error bar is ±1 SD across
those folds. The folds share training rows, so read the bars as the spread of the
estimate, not as a confidence interval.

In [ ]:
fig = config.FIG_DIR / "accuracy_vs_distance.png"
display(Image(str(fig))) if fig.exists() else print("run `python plot.py`")

In [ ]:
default = probe["default_persona"]
dist = axis["per_layer"][BEST]["distance"]
transfer = probe["per_layer"][BEST]["transfer"]
pl = probe["per_layer"][BEST]

rows = []
for p in personas.PERSONAS:
    pid = p["id"]
    if pid not in dist:
        continue
    if pid == default:
        acc, sd = pl["acc_default_heldout"], pl["acc_default_heldout_sd"]
    else:
        t = transfer.get(pid, {})
        acc, sd = t.get("acc_heldout_scenarios"), t.get("acc_heldout_scenarios_sd")
    rows.append({
        "persona": pid,
        "kind": "control" if p["is_control"] else ("trained" if pid == default else "persona"),
        "intuited_rank": p["distance_rank"],
        "axis_distance": round(dist[pid], 3),
        "accuracy": None if acc is None else round(acc, 3),
        "sd_folds": None if sd is None else round(sd, 3),
        "vs_chance": None if acc is None else round(acc / probe["chance"], 1),
    })

res = pd.DataFrame(rows).sort_values("axis_distance").reset_index(drop=True)
res["measured_rank"] = range(len(res))
display(res)
print(f"accuracy = mean over {CV['n_folds']} topic folds; sd_folds = SD of the "
      f"per-fold values (the error bars in the figure)")

personas_only = res[res["kind"] != "control"]
if len(personas_only) > 2 and personas_only["accuracy"].notna().all():
    r = personas_only["axis_distance"].corr(personas_only["accuracy"], method="spearman")
    print(f"Spearman(axis distance, accuracy) over personas only: {r:.3f}")
    print("negative = accuracy falls as the persona moves away from the assistant")

## The control

`bare_template` sends no system message, so Qwen's chat template supplies its own
— short, and naming the vendor. The written personas are ~50 words.

If the control sits near the anchor, prompt length is not what the axis is
measuring. If it sits far from the anchor, the axis is picking up prompt length or
template effects as well as persona, and the headline figure has a confound in it.
Either way it is a finding, not a nuisance.

In [ ]:
ctrl = axis["control_persona"]
per_layer_d = {int(l): axis["per_layer"][str(l)]["distance"] for l in axis["layers"]}
ctrl_by_layer = pd.DataFrame({
    "layer": list(per_layer_d),
    f"{ctrl} distance": [round(d[ctrl], 3) for d in per_layer_d.values()],
    "nearest persona": [
        min((k for k in d if k != ctrl), key=lambda k: abs(d[k] - d[ctrl]))
        for d in per_layer_d.values()
    ],
}).set_index("layer")
display(ctrl_by_layer)

mean_d = sum(d[ctrl] for d in per_layer_d.values()) / len(per_layer_d)
verdict = ("close to the anchor — prompt length is not driving the axis"
           if abs(mean_d) < 0.25 else
           "NOT close to the anchor — the axis may be measuring prompt length "
           "or template effects, not persona alone")
print(f"{ctrl} mean distance across layers: {mean_d:.3f}  ->  {verdict}")

## Intuited vs measured order

The persona ordering in `personas.py` was a prior. This is the measurement.

In [ ]:
cmp = res[res["kind"] != "control"].copy()
cmp["intuited_order"] = cmp["intuited_rank"].rank().astype(int) - 1
cmp["measured_order"] = range(len(cmp))
cmp["shift"] = cmp["measured_order"] - cmp["intuited_order"]
display(cmp[["persona", "intuited_order", "measured_order", "shift", "axis_distance"]])

moved = cmp[cmp["shift"] != 0]
print(f"{len(moved)} persona(s) moved from their intuited position"
      if len(moved) else "measured order matches the intuited order exactly")

## Layer choice

The layer is picked on default-persona accuracy alone. Picking it on transfer
accuracy would select for the answer being measured.

In [ ]:
fig = config.FIG_DIR / "accuracy_by_layer.png"
display(Image(str(fig))) if fig.exists() else print("run `python plot.py`")

by_layer = pd.DataFrame([
    {"layer": l,
     "default_heldout": round(probe["per_layer"][str(l)]["acc_default_heldout"], 3),
     "sd_folds": (None if probe["per_layer"][str(l)]["acc_default_heldout_sd"] is None
                  else round(probe["per_layer"][str(l)]["acc_default_heldout_sd"], 3)),
     **{pid: (None if t["acc_heldout_scenarios"] is None
              else round(t["acc_heldout_scenarios"], 3))
        for pid, t in probe["per_layer"][str(l)]["transfer"].items()}}
    for l in probe["layers"]
]).set_index("layer")
display(by_layer)

# Per-fold accuracies at the chosen layer: the raw material behind the error bars.
folds = pd.DataFrame(
    {default: probe["per_layer"][BEST]["acc_default_heldout_folds"],
     **{pid: t["acc_heldout_scenarios_folds"]
        for pid, t in probe["per_layer"][BEST]["transfer"].items()}},
    index=[f"fold {k}" for k in range(CV["n_folds"])],
).round(3)
display(folds)

## Where the default probe fails

Confusion is more informative than accuracy: emotions that trade with each other
(nervous/afraid, sad/guilty) are a different failure from emotions that collapse
into one attractor class.

In [ ]:
fig = config.FIG_DIR / "confusion_matrix.png"
display(Image(str(fig))) if fig.exists() else print("run `python plot.py`")

In [ ]:
import numpy as np

classes = probe["per_layer"][BEST]["classes"]
cm = np.array(probe["per_layer"][BEST]["confusion_matrix"], dtype=float)

per_class = pd.DataFrame({
    "emotion": classes,
    "n": cm.sum(axis=1).astype(int),
    "recall": (np.diag(cm) / np.where(cm.sum(axis=1) > 0, cm.sum(axis=1), 1)).round(3),
}).sort_values("recall")
display(per_class.reset_index(drop=True))

off = [(classes[i], classes[j], int(cm[i, j]))
       for i in range(len(classes)) for j in range(len(classes))
       if i != j and cm[i, j] > 0]
print("most common confusions (true -> predicted):")
for true, pred, n in sorted(off, key=lambda t: -t[2])[:8]:
    print(f"  {n:3d}  {true} -> {pred}")

---
Figures and their `.csv` twins are in `outputs/figures/`.